# Credit Risk Model — Exploratory Data Analysis

**Dataset**: Xente eCommerce transaction data  
**Goal**: Understand the data, validate feature engineering assumptions, and justify the RFM-based proxy default label.

## Contents
1. Load & Inspect
2. Missing Values & Data Quality
3. Distribution of Numerical Features
4. Distribution of Categorical Features
5. Time Series Analysis
6. Outlier Detection
7. Fraud Analysis
8. Correlation Analysis
9. RFM Feature Engineering & Clustering
10. Proxy Label Validation
11. Key Insights Summary


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = '../data/raw/data.csv'
print('Libraries loaded')

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['TransactionStartTime'])
print(f'Shape: {df.shape}')
print(f'Date range: {df.TransactionStartTime.min()} to {df.TransactionStartTime.max()}')
print(f'Unique customers: {df.CustomerId.nunique()}')
print(f'Unique products: {df.ProductId.nunique()}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('Data types:')
print(df.dtypes)
print('\nSummary statistics:')
df.describe()

## 2. Missing Values & Data Quality

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])
print('\nNo missing values!' if missing.sum() == 0 else f'Total missing: {missing.sum()}')

# Duplicate check
dupes = df.duplicated(subset='TransactionId').sum()
print(f'\nDuplicate TransactionIds: {dupes}')

# Amount vs Value consistency
df['amount_value_match'] = df['Amount'].abs().round(2) == df['Value'].round(2)
print(f'Amount/Value mismatch: {(~df.amount_value_match).sum()} rows')

## 3. Distribution of Numerical Features

In [ ]:
numerical_cols = ['Amount', 'Value']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Amount distribution
debits = df[df['Amount'] > 0]['Amount']
axes[0][0].hist(debits.clip(upper=debits.quantile(0.99)), bins=50, color='steelblue', edgecolor='white')
axes[0][0].set_title('Transaction Amount (debits, clipped at 99th pct)')
axes[0][0].set_xlabel('Amount')

# Log-transformed amount
axes[0][1].hist(np.log1p(debits), bins=50, color='coral', edgecolor='white')
axes[0][1].set_title('Log(Amount + 1) - reveals right-skew')
axes[0][1].set_xlabel('log(Amount)')

# Value distribution
axes[1][0].hist(df['Value'].clip(upper=df['Value'].quantile(0.99)), bins=50, color='mediumseagreen', edgecolor='white')
axes[1][0].set_title('Transaction Value (clipped at 99th pct)')
axes[1][0].set_xlabel('Value')

# Transactions per customer
txn_per_customer = df.groupby('CustomerId').size()
axes[1][1].hist(txn_per_customer.clip(upper=txn_per_customer.quantile(0.99)), bins=40, color='mediumpurple', edgecolor='white')
axes[1][1].set_title('Transactions per Customer (clipped at 99th pct)')
axes[1][1].set_xlabel('Count')

plt.suptitle('Numerical Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Amount skewness: {debits.skew():.2f} (highly right-skewed)')
print(f'Median transaction: {debits.median():.2f}')
print(f'Mean transactions/customer: {txn_per_customer.mean():.1f}')
print(f'Max transactions/customer: {txn_per_customer.max()}')

## 4. Distribution of Categorical Features

In [ ]:
cat_cols = ['ProductCategory', 'ChannelId', 'PricingStrategy', 'CurrencyCode']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, col in zip(axes.flat, cat_cols):
    counts = df[col].value_counts().head(10)
    ax.barh(counts.index[::-1], counts.values[::-1], color='steelblue')
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel('Transaction Count')

plt.suptitle('Categorical Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

for col in cat_cols:
    print(f'{col}: {df[col].nunique()} unique values')

## 5. Time Series Analysis

In [ ]:
df['date'] = df['TransactionStartTime'].dt.date
df['hour'] = df['TransactionStartTime'].dt.hour
df['dayofweek'] = df['TransactionStartTime'].dt.day_name()
df['month'] = df['TransactionStartTime'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Daily volume
daily = df.groupby('date').size()
axes[0][0].plot(daily.index, daily.values, linewidth=0.8, color='steelblue')
axes[0][0].set_title('Daily Transaction Volume')
axes[0][0].tick_params(axis='x', rotation=45)

# Hourly distribution
hourly = df.groupby('hour').size()
axes[0][1].bar(hourly.index, hourly.values, color='coral', edgecolor='white')
axes[0][1].set_title('Transactions by Hour of Day')
axes[0][1].set_xlabel('Hour')

# Day of week
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = df.groupby('dayofweek').size().reindex(day_order)
axes[1][0].bar(range(7), dow_counts.values, color='mediumseagreen', edgecolor='white')
axes[1][0].set_xticks(range(7))
axes[1][0].set_xticklabels([d[:3] for d in day_order], rotation=45)
axes[1][0].set_title('Transactions by Day of Week')

# Monthly
monthly = df.groupby('month').size()
axes[1][1].bar(monthly.index, monthly.values, color='mediumpurple', edgecolor='white')
axes[1][1].set_title('Transactions by Month')
axes[1][1].set_xlabel('Month')

plt.suptitle('Time-Based Transaction Patterns', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Amount boxplot
axes[0].boxplot(df[df['Amount'] > 0]['Amount'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0].set_title('Amount Outliers')
axes[0].set_ylabel('Amount')

# Value boxplot
axes[1].boxplot(df['Value'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='coral', alpha=0.6))
axes[1].set_title('Value Outliers')
axes[1].set_ylabel('Value')

# Transactions per customer boxplot
txn_counts = df.groupby('CustomerId').size()
axes[2].boxplot(txn_counts, vert=True, patch_artist=True,
                boxprops=dict(facecolor='mediumseagreen', alpha=0.6))
axes[2].set_title('Transactions per Customer Outliers')
axes[2].set_ylabel('Count')

plt.suptitle('Outlier Detection via Box Plots', fontsize=13)
plt.tight_layout()
plt.show()

# IQR-based outlier count
for col, series in [('Amount', df[df['Amount']>0]['Amount']), ('Value', df['Value'])]:
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()
    print(f'{col}: {outliers} outliers ({outliers/len(series)*100:.1f}%)')

## 7. Fraud Analysis

In [ ]:
fraud_rate = df['FraudResult'].mean()
print(f'Overall fraud rate: {fraud_rate:.4f} ({fraud_rate*100:.2f}%)')
print(f'Fraud transactions: {df.FraudResult.sum()} / {len(df)}')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

fraud_by_cat = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False)
axes[0].barh(fraud_by_cat.index[::-1], fraud_by_cat.values[::-1], color='salmon')
axes[0].set_title('Fraud Rate by Product Category')
axes[0].set_xlabel('Fraud Rate')
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

fraud_amounts = df[df['FraudResult'] == 1]['Amount'].abs()
normal_amounts = df[df['FraudResult'] == 0]['Amount'].abs()
axes[1].hist(np.log1p(normal_amounts.clip(upper=normal_amounts.quantile(0.99))),
             bins=40, alpha=0.6, label='Normal', color='steelblue')
axes[1].hist(np.log1p(fraud_amounts.clip(upper=fraud_amounts.quantile(0.99))),
             bins=40, alpha=0.6, label='Fraud', color='salmon')
axes[1].set_title('Log(Amount) - Fraud vs Normal')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
from src.data_processing import build_feature_matrix

features = build_feature_matrix(df)

corr_cols = ['Recency', 'Frequency', 'Monetary', 'TotalTransactionAmount',
             'AvgTransactionAmount', 'TransactionCount', 'StdTransactionAmount',
             'FraudRate', 'UniqueProducts', 'UniqueChannels', 'NightTxnRatio', 'is_high_risk']

# Only use columns that exist
corr_cols = [c for c in corr_cols if c in features.columns]
corr = features[corr_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

target_col = 'is_high_risk'
print(f'\nCorrelation with {target_col}:')
print(corr[target_col].drop(target_col).sort_values(key=abs, ascending=False).round(3))

## 9. RFM Feature Engineering & Clustering

Since the dataset has no ground-truth default label, we engineer a **proxy target variable** using RFM segmentation:
- **Recency**: Days since last transaction (higher = more dormant = riskier)
- **Frequency**: Number of transactions (lower = less engaged = riskier)
- **Monetary**: Total spend (lower = less valuable = riskier)

K-Means clustering (k=3) groups customers, and the cluster with highest recency + lowest frequency/monetary is labelled `is_high_risk=1`.

In [ ]:
from src.data_processing import compute_rfm, assign_high_risk_label

rfm = compute_rfm(df)
rfm_labeled = assign_high_risk_label(rfm)

print('RFM Cluster Summary:')
print(rfm_labeled.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(1))
print(f'\nProxy high-risk rate: {rfm_labeled.is_high_risk.mean():.2%}')
print(f'High-risk customers: {rfm_labeled.is_high_risk.sum()} / {len(rfm_labeled)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {0: 'salmon', 1: 'steelblue', 2: 'mediumseagreen'}

for cluster in sorted(rfm_labeled['Cluster'].unique()):
    subset = rfm_labeled[rfm_labeled['Cluster'] == cluster]
    risk = 'HIGH RISK' if subset.is_high_risk.iloc[0] == 1 else 'low risk'
    label = f'Cluster {cluster} ({risk})'
    c = colors.get(cluster, 'gray')
    axes[0].scatter(subset['Recency'], subset['Frequency'], alpha=0.4, s=10, label=label, color=c)
    axes[1].scatter(subset['Recency'], np.log1p(subset['Monetary']), alpha=0.4, s=10, color=c)
    axes[2].scatter(subset['Frequency'], np.log1p(subset['Monetary']), alpha=0.4, s=10, color=c)

axes[0].set_xlabel('Recency (days)'); axes[0].set_ylabel('Frequency'); axes[0].set_title('Recency vs Frequency')
axes[1].set_xlabel('Recency (days)'); axes[1].set_ylabel('log(Monetary)'); axes[1].set_title('Recency vs Monetary')
axes[2].set_xlabel('Frequency'); axes[2].set_ylabel('log(Monetary)'); axes[2].set_title('Frequency vs Monetary')
axes[0].legend(fontsize=8)

plt.suptitle('RFM Cluster Visualization', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 10. Proxy Label Validation

In [ ]:
feature_list = [c for c in ['Recency', 'Frequency', 'Monetary', 'AvgTransactionAmount',
                              'FraudRate', 'NightTxnRatio'] if c in features.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, feat in zip(axes.flat, feature_list):
    good = features[features['is_high_risk'] == 0][feat]
    bad  = features[features['is_high_risk'] == 1][feat]
    clip_val = features[feat].quantile(0.99)
    ax.hist(good.clip(upper=clip_val), bins=30, alpha=0.6, label='Low Risk (0)', color='steelblue', density=True)
    ax.hist(bad.clip(upper=clip_val),  bins=30, alpha=0.6, label='High Risk (1)', color='salmon',   density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions: Low Risk vs High Risk', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('=== Mean values by risk label ===')
print(features.groupby('is_high_risk')[feature_list].mean().round(2).T)

## 11. Key Insights Summary

---

### Insight 1: Transaction amounts are highly right-skewed with significant outliers
The raw `Amount` distribution is heavily right-skewed (skewness > 10). A small number of customers account for disproportionately large transaction values — one cluster has a mean monetary value over 50 million. Log transformation reveals a more interpretable bimodal pattern. These extreme values must be handled during feature engineering (either capping or log-scaling) to prevent them from dominating model training.

---

### Insight 2: Customer engagement is extremely unequal — most customers transact very infrequently
The distribution of transactions per customer is heavily right-skewed. The majority of customers have fewer than 10 transactions in the dataset, while a small group of power users has hundreds. This bimodal engagement pattern is the behavioral signal we exploit to engineer the proxy default label: disengaged customers (low frequency, high recency) form a natural high-risk cluster.

---

### Insight 3: Fraud is rare (< 2%) but concentrated in specific product categories
The overall fraud rate is under 2%, creating a significant class imbalance. However, fraud is not random — specific product categories have materially higher fraud rates. This means `FraudRate` per customer is a potentially informative feature for credit risk, since customers who have experienced fraud transactions may represent different risk profiles.

---

### Insight 4: RFM clustering clearly separates three distinct customer segments
K-Means clustering on scaled RFM features produces well-separated clusters visible in the scatter plots. The high-risk cluster (is_high_risk=1) is characterized by: high Recency (avg 60+ days since last transaction), low Frequency (avg < 5 transactions), and low Monetary value. This is behaviorally consistent with credit risk intuition — dormant, low-value customers are more likely to be financially disengaged.

---

### Insight 5: Recency and Frequency are the strongest predictors of the proxy label
The correlation matrix shows that Recency has the strongest positive correlation with is_high_risk, while Frequency and Monetary have the strongest negative correlations. This confirms our proxy label construction is internally consistent. However, the near-perfect model AUC (1.0) indicates the model is learning cluster membership rather than genuine default risk — a known limitation of proxy-label approaches that must be disclosed.

---

**Modeling implications**: Log-transform or cap Amount/Monetary features. Use class_weight='balanced' for all classifiers due to the 39% proxy default rate. Include Logistic Regression as an interpretable Basel II-compliant baseline alongside gradient boosting models.